# The query interface — feature demo

One cell per feature, on the WaterTAP seawater-RO model.

**Start the server** (from the repo root, wait for the first solve to ingest data):

```bash
uv run acquirium server --config deployments/WATERTAP/models/seawater-ro/acquirium.toml 
```

**The interface in one paragraph** — `acq.query()` builds a pattern out of three
verbs (`entity`, `related`, `measurement`). Every attribute (medium, unit,
process, ...) is one vocabulary shared by filtering (`where`), projection
(`include`) and faceting (`options`/`facets`); `Not()` negates a value.
Multi-hop traversal runs as a client-side BFS, so SPARQL never sees
join-explosive chains, and `nearest=` keeps the closest match. Attribute
predicates are hidden from generic traversal by default, and aliases default to
the class name you typed.

In [ ]:
from acquirium import Acquirium                     # constructor now waits for /health
from acquirium.Client.explore import Not, hidden_predicates

acq = Acquirium(server_url="localhost", server_port=8000)

## Build patterns

`entity(cls)` — class as URI or free text (server-resolved); the alias
defaults to what you typed.

In [ ]:
acq.query().entity("pump").metadata()

`alias()` names the current node; `uri=` pins an instance (CURIEs work).

In [ ]:
acq.query().entity(uri="wbs:RO").alias("ro").metadata()

`related(cls)` finds related entities — by default the *nearest* match
within 3 hops of any non-hidden predicate.

In [ ]:
acq.query().entity("pump").related("tank",nearest=False,direction="downstream").metadata()

`via=` restricts traversal to a predicate (repeatable up to
`max_depth`), a list of predicates, or `"any"`; `nearest=False` returns all
matches, `max_depth=0` opts into unbounded.

In [ ]:
(acq.query().entity("System")
 .related("Equipment", via="hasMember/connectedTo", nearest=False)
 .metadata())

`direction="upstream"/"downstream"` walks the s223 piping topology; the
step patterns each direction infers are inspectable constants in
`explore.directions` and can be passed to `via=` for nearest searches.

In [ ]:
(acq.query().entity(uri="wbs:RO")
 .related("pump", direction="upstream")
 .metadata())

## Measurements

`measurement()` attaches data-bearing points — the source's own plus its
connection points' (`include_connection_points=False` for own only);
keyword attributes filter inline, `Not()` excludes, lists mean OR.

In [ ]:
from acquirium.Client.explore import Not
(acq.query().entity(uri="wbs:RO").alias("ro")
 .measurement(alias="feed").where(quantity_kind="mass flow rate", medium=Not("brine"))
 .metadata())

In [ ]:
acq.query().entity("pump").related("Pressure Exchanger").measurement(frm="*").metadata().head(10)

On an empty query, `measurement()` is the root form: every registered
stream in the plant (`frm="*"` / `frm=["a", "b"]` attach per-entity).

In [ ]:
acq.query().measurement(quantity_kind="pressure").metadata()

`measurement(direction=..., nearest=True)` finds the closest up/downstream
measurement matching the filters.

In [ ]:
(acq.query().entity(uri="wbs:P1").alias("p1")
 .measurement(direction="downstream", nearest=True, quantity_kind="pressure")
 .options("quantity_kind"))

## Filter, project, shape

`where()` filters any node by alias (`target=`), same attribute vocabulary
everywhere.

In [ ]:
## process is its own resolver kind, so equipment classes never outrank it
(acq.query().entity("Equipment").where(process="reverse osmosis")
 .metadata())

`include()` adds `alias.attr` columns (placed right after their node's
column); `required=True` drops rows lacking the attribute.

In [ ]:
(acq.query().entity(uri="wbs:RO").include("process").measurement(alias="m")
 .include("quantity_kind", "unit")
 .metadata())

`drop()` keeps a node in the pattern but out of the output (rows
deduplicate accordingly); `refocus()` moves the pointer back to an alias.

In [ ]:
(acq.query().entity(uri="wbs:pretreatment-system").drop()
 .related("equipment").measurement(alias="sensor")
 .metadata())

`with_columns()` merges the two: plain specs include, `"-"`-prefixed
specs drop; dotted `"alias.attr"` targets any node's attribute.

In [ ]:
(acq.query().entity(uri="wbs:RO").alias("ro")
 .measurement(alias="m")
 .with_columns("m.quantity_kind", "m.unit", "-ro")
 .metadata())

`include()` and `drop()` are inverses — each accepts the other's
vocabulary, so any column decision can be reversed later in the chain
(here: un-drop `ro`, un-include `unit`).

In [ ]:
q = (acq.query().entity(uri="wbs:RO").alias("ro").drop()
     .measurement(alias="m").include("unit"))
q.include("ro").drop("unit").metadata()

## Faceted exploration

`options(attr)` — the distinct values of one attribute across the current
matches, counted client-side.

In [ ]:
acq.query().measurement().options("quantity_kind")

`facets()` — every applicable attribute at once, falling back to
model-wide then ontology vocabulary when the pattern is empty.

In [ ]:
acq.query().entity(uri="wbs:RO").alias("RO").measurement().facets()

## Data

`data()` returns the lazy DataObject; `dataframe(shape="wide")` puts
`time` first and value columns in alphabetical order; `convert_to` resolves
the target unit *jointly with the source* so only convertible matches win.

In [ ]:
d = (acq.query().measurement(alias="tds", quantity_kind="mass concentration")
     .data(cast_value="float"))
d.dataframe(shape="wide").tail(3)

In [ ]:
d.convert_to("mg/L").dataframe(shape="wide").tail(3)

## Guard rails

Attribute predicates, `subClassOf`, `hasProperty`, and `s223:cnx` are hidden
from `via="any"` traversal by default (`hide()`/`unhide()` adjust); every
attribute-taking method documents the attribute table in its docstring
(`help(q.where)`); and inspect any query with `to_sparql()` before running.

In [ ]:
sorted(hidden_predicates())

In [ ]:
print(acq.query().entity("pump").measurement(alias="m").to_sparql())